# Phase 1: Tokenization & Text Preprocessing

Welcome to the first notebook of your NLP learning journey! This repository is designed to be an educational resource. Rather than just running code, we walk through the algorithmic choices and the practical implementation details of each concept.

## What is Preprocessing, and Why Do We Need It?
Raw text is a sequence of characters. Machine learning algorithms, however, operate on numbers. The bridge between raw text and mathematical vectors is **text preprocessing**.

In this phase, we explore the following key stages of the preprocessing pipeline:
- **Tokenization**: Breaking continuous strings into discrete units (words, subwords, or sentences).
- **Stemming & Lemmatization**: Reducing words to their base or root forms to collapse vocabulary size.
- **Stopword Filtering**: Removing high-frequency, low-semantic-value words.
- **Regex Cleaning**: Stripping URLs, HTML entities, and formatting noise.

Let's get started!

---  
# Part 1: Word & Sentence Tokenization

Tokenization is the process of splitting text into a sequence of smaller, meaningful units called tokens. Let's compare various word tokenization approaches side-by-side on the same sample text to see their differences.

## 1.1 Whitespace & Regex Tokenization from Scratch

First, let's implement basic tokenizers from scratch using standard Python. This will help us understand where the edge cases live.

In [ ]:
def tokenize_whitespace(text):
    """
    Splits text strictly by whitespace characters (spaces, tabs, newlines).
    """
    return text.split()

import re
def tokenize_regex(text):
    """
    Splits text using regular expressions, separating punctuation from words
    while keeping contractions (e.g., "doesn't") together.
    """
    pattern = r"\w+(?:'\w+)?|[!\"#$%&'()*+,\-./:;<=>?@[\\\]^_`{|}~]"
    return re.findall(pattern, text)

## 1.2 Side-by-Side Tokenizer Comparison

Let's import **NLTK**, **spaCy**, and the **Hugging Face transformers** tokenizer to see how they process the same sentence compared to our scratch implementations.

In [ ]:
import nltk
import spacy
from transformers import GPT2Tokenizer
import pandas as pd

# Ensure models are loaded
nlp = spacy.load("en_core_web_sm")
gpt_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

sample_text = "NLP is state-of-the-art! However, it doesn't always work perfectly."

# Get outputs from all tokenizers
tokens_whitespace = tokenize_whitespace(sample_text)
tokens_regex = tokenize_regex(sample_text)
tokens_nltk = nltk.word_tokenize(sample_text)
tokens_spacy = [token.text for token in nlp(sample_text)]
tokens_gpt2 = gpt_tokenizer.tokenize(sample_text)

# Print them formatted neatly for visual comparison
print(f"Original Text: \"{sample_text}\"\n")
print(f"{'Method':<25} | {'Tokens'}")
print("-" * 80)
print(f"{'Whitespace (Scratch)':<25} | {tokens_whitespace}")
print(f"{'Regex (Scratch)':<25} | {tokens_regex}")
print(f"{'NLTK Word Tokenizer':<25} | {tokens_nltk}")
print(f"{'spaCy Tokenizer':<25} | {tokens_spacy}")
print(f"{'GPT-2 BPE Tokenizer':<25} | {tokens_gpt2}")

## 1.3 Explaining the Tokenizer Outputs

By examining the table above, we can spot distinct design philosophies between these tokenizers:

1. **Punctuation Handling (`state-of-the-art!`, `perfectly.`)**
   - **Whitespace**: Keeps the exclamation mark attached (`state-of-the-art!`) and the period attached (`perfectly.`). This is problematic because `perfectly.` and `perfectly` will be treated as different vocabulary index entries.
   - **Regex, NLTK, & spaCy**: Successfully separate punctuation (`!`, `.`) into independent tokens.
   - **GPT-2 BPE**: Splits punctuation as well, but notes spaces using a special character `Ġ` (representing a space prefix, like `Ġperfectly` and `.`).

2. **Contractions (`doesn't`)**
   - **Whitespace**: Keeps it intact as `doesn't`.
   - **Regex (Scratch)**: Keeps it intact because our pattern `\w+(?:'\w+)?` matches the apostrophe and character `'t`.
   - **NLTK**: Splits it into `does` and `n't`. This is Penn Treebank style, isolating the negating particle `not` (`n't`) so downstream models can capture negation clearly.
   - **spaCy**: Splits it morphologically into `does` and `n't` as well, representing `does` (verb) and `not` (adverb).
   - **GPT-2 BPE**: Splits it into `doesn` and `'t` based on common character-pair frequencies.

3. **Hyphenation (`state-of-the-art`)**
   - **NLTK & Regex**: Keep `state-of-the-art` as a single token.
   - **spaCy**: Splits it into 5 distinct tokens: `state`, `-`, `of`, `-`, `the`, `-`, `art`. spaCy avoids grouping hyphenated structures to maintain granular POS tagging capability.
   - **GPT-2 BPE**: Splits it into sub-units (`state`, `-`, `of`, `-`, `the`, `-`, `art`) because BPE constructs words out of frequent characters rather than morphological rules.

## 1.4 Sentence Tokenization

Let's look at sentence splitting. A basic splitter might split on every period, but abbreviations can cause false sentence boundaries. Let's compare NLTK's *Punkt* (unsupervised heuristic model) and spaCy's dependency parser sentence segmenter.

In [ ]:
sentence_sample = "Dr. Smith bought a laptop. It cost $1,000.50. What a deal!"

nltk_sentences = nltk.sent_tokenize(sentence_sample)
spacy_sentences = [sent.text for sent in nlp(sentence_sample).sents]

print("Original:", sentence_sample)
print("-" * 80)
print("NLTK sentences:")
for i, s in enumerate(nltk_sentences):
    print(f"  {i+1}: {s}")

print("\nspaCy sentences:")
for i, s in enumerate(spacy_sentences):
    print(f"  {i+1}: {s}")

### Sentence Splitting Explanation:
- Both **NLTK** and **spaCy** successfully avoid splitting at `Dr.` because they recognize it is an abbreviation. They also avoid splitting at `$1,000.50` because the period is surrounded by digits.
- They split correctly at `. ` and `! ` into 3 sentences. spaCy uses its parser to detect boundaries based on subject-verb dependencies, whereas NLTK relies on punctuation heuristics.

---  
# Part 2: Normalization & Vocabulary Reduction

Once text is tokenized, we normalize it to reduce vocabulary size. This involves case folding, Unicode normalization, stemming, and lemmatization.

## 2.1 Case Folding & Unicode Normalization

- **Case Folding**: Converting all text to lowercase. While useful for grouping words like `Apple` and `apple` together, it can lead to ambiguity (e.g., `US` the country vs. `us` the pronoun).
- **Unicode Normalization**: Accented characters can be represented in multiple ways in Unicode (e.g., standard `é` vs. `e` combined with acute accent `́`). Normalizing ensures uniform byte strings.

In [ ]:
import unicodedata

word1 = "Café" # Represented as a single character 'é' (NFC)
word2 = "Cafe\u0301" # Represented as 'e' + combining accent (NFD)

print(f"word1 == word2: {word1 == word2}")
print(f"word1 bytes: {word1.encode('utf-8')}")
print(f"word2 bytes: {word2.encode('utf-8')}")

# Normalize to NFC (Canonical Composition)
norm_word1 = unicodedata.normalize('NFC', word1)
norm_word2 = unicodedata.normalize('NFC', word2)
print(f"Normalized word1 == word2: {norm_word1 == norm_word2}")

## 2.2 Stemming vs. Lemmatization Side-by-Side

Let's compare three stemmers: NLTK's **Porter**, **Snowball** (a slightly improved Porter), and **Lancaster** (highly aggressive), against spaCy's **Lemmatizer**.

In [ ]:
from nltk.stem import PorterStemmer, SnowballStemmer, LancasterStemmer

porter = PorterStemmer()
snowball = SnowballStemmer("english")
lancaster = LancasterStemmer()

words = ["connecting", "connections", "connected", "studies", "studying", "was", "am", "better", "mice", "wolves"]

norm_data = []
for w in words:
    p_stem = porter.stem(w)
    s_stem = snowball.stem(w)
    l_stem = lancaster.stem(w)
    
    doc_w = nlp(w)
    lemma = doc_w[0].lemma_
    
    norm_data.append({
        "Original": w,
        "Porter Stem": p_stem,
        "Snowball Stem": s_stem,
        "Lancaster (Aggressive)": l_stem,
        "spaCy Lemma": lemma
    })

df_norm = pd.DataFrame(norm_data)
df_norm

### Explaining the Normalization Table:

1. **Regular Suffixes (`connecting`, `connections`, `connected`)**
   - All stemmers successfully strip `-ing`, `-ions`, `-ed` to reduce them to `connect`.
   - spaCy lemmatizer reduces them to `connect` (or `connection` if marked as noun), showing similar grouping capability.

2. **Spelling Heuristics (`studies`, `studying`)**
   - **Porter & Snowball**: Convert `studies` to `studi` and `studying` to `studi`. This groups them under a single root, but `studi` is not a real dictionary word.
   - **Lancaster**: Aggressively strips endings, reducing `studies` to `study` and `studying` to `study`.
   - **spaCy Lemmatizer**: Correctly uses morphological rules to resolve both to `study`.

3. **Irregular Forms (`was`, `am`)**
   - **Stemmers**: Fail to recognize semantic roots, producing `wa` (Porter/Snowball) or `was`/`am` (unmodified). They only strip character patterns.
   - **Lemmatizer**: Correctly maps auxiliary verbs back to their base infinitive `be`.

4. **Plural Nouns (`mice`, `wolves`)**
   - **Stemmers**: Keep `mice` unchanged because there is no common plural suffix to strip. `wolves` is chopped to `wolv`.
   - **Lemmatizer**: Correctly looks up the dictionary forms to yield `mouse` and `wolf`.

#### Over-stemming vs. Under-stemming
- **Over-stemming**: Cutting off too much, merging words with different meanings (e.g., Lancaster stemmer cutting `organization` and `organs` to `org`).
- **Under-stemming**: Failing to strip enough, leaving related words separate (e.g., Porter stemmer keeping `alumnus` and `alumni` separate).

---  
# Part 3: Byte Pair Encoding (BPE) subword details

Let's trace how the Byte Pair Encoding (BPE) subword tokenizer splits unknown or rare words compared to standard vocabulary items.

In [ ]:
test_phrases = [
    "HuggingFace library",
    "electromagnetism",
    "antidisestablishmentarianism"
]

for phrase in test_phrases:
    sub_tokens = gpt_tokenizer.tokenize(phrase)
    print(f"Phrase: {phrase:<30} | Subwords: {sub_tokens}")

### Subword Tokenization Explanation:
- BPE does not have a vocabulary item for the composite word `HuggingFace`. Instead, it breaks it down into familiar subword pieces: `['H', 'ugging', 'F', 'ace']`.
- Very long or rare words like `antidisestablishmentarianism` are split into many high-frequency parts: `['ant', 'id', 'is', 'est', 'ab', 'lish', 'ment', 'arian', 'ism']`.
- This approach guarantees that the model has **zero out-of-vocabulary (OOV)** words, as any word can be decomposed down to individual characters if needed.

---  
# Part 4: Hands-On Exercises

Complete these three exercises to build your preprocessing pipeline. Check the comments for directions!

## Exercise 4.1: Stemming vs. Lemmatization Comparison

### Objective
Process a small batch of sentences and extract the stemming/lemmatization differences. Focus on identifying:
- **Over-stemming**: Where the stemmer cuts off too much (e.g., resolving unrelated words to the same root, losing meaning).
- **Under-stemming**: Where the stemmer fails to group related words to the same root.

In [ ]:
# Define sentences with complex word forms
sentences = [
    "The children are playing happily in the garden, having fun with their toys.",
    "The studies showed that studying regularly produces better study results.",
    "He was running, jumps over the fence, and then walked slowly home.",
    "The items were bought at a discount, but some were broken or damaged."
]

# TODO: Tokenize each sentence using NLTK, compute the stem (NLTK Porter) and lemma (spaCy) for each token.
# Compile the results into a single pandas DataFrame for comparison and display it.

## Exercise 4.2: Noisy Social Media Cleaning Pipeline

### Objective
Build a robust cleaning function using regular expressions to clean messy tweets before tokenization.

In [ ]:
tweets = [
    "Check out this amazing article! https://t.co/xyz123 @user123 #NLP #AI is the future!!!",
    "RT @another_user: Python is &lt;great&gt; for data science. Go to http://python.org",
    "I    love   natural    language    processing... it's so cool!!!"
]

def clean_tweet(text):
    """
    Cleans a tweet by:
    1. Removing URLs (http:// or https:// and any following non-whitespace characters).
    2. Removing user mentions (@username).
    3. Stripping/converting HTML entities (like converting '&lt;' to '<', '&gt;' to '>', and '&amp;' to '&').
    4. Normalizing whitespace (replacing tabs, newlines, and multiple spaces with a single space, stripping boundary spaces).
    """
    # TODO: Implement the cleaning logic
    pass

for tweet in tweets:
    print("Original:", tweet)
    print("Cleaned: ", clean_tweet(tweet))
    print("-" * 50)

## Exercise 4.3: Complete Preprocessing Pipeline

### Objective
Combine cleaning, tokenization, stopword removal, and lemmatization into a single production-like function.

In [ ]:
import spacy
nlp = spacy.load("en_core_web_sm")

def preprocess_text(text):
    """
    Takes raw text, cleans it (removes URLs/mentions, normalizes whitespace),
    tokenizes it, removes stopwords and punctuation, and returns a list of lemmatized tokens in lowercase.
    
    Args:
        text (str): The input raw string.
        
    Returns:
        list: Cleaned, tokenized, and lemmatized token strings.
    """
    # TODO: Implement complete preprocessing pipeline using spaCy and clean_tweet
    pass

In [ ]:
raw_doc = """The Quick Brown Fox jumps over the lazy dog! 
Visit http://example.com/fox for more details. @fox_fan club."""

print("Preprocessed:", preprocess_text(raw_doc))
# Expected output should be a list of lowercased lemmas, e.g.:
# ['quick', 'brown', 'fox', 'jump', 'lazy', 'dog', 'visit', 'detail', 'club']